In [19]:
import math, random, time, json
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
import matplotlib.pyplot as plt

import torch, random, numpy as np
from torchvision import datasets, transforms
from torch.utils.data import random_split, DataLoader


## **Get the Dataset**



In [17]:

# Fix seeds
SEED = 0
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

#  Download 
root = "./data"
train_full = datasets.CIFAR10(root, train=True,  download=True, transform=transforms.ToTensor())
test_set   = datasets.CIFAR10(root, train=False, download=True, transform=transforms.ToTensor())

#  Split 45k / 5k with a seeded generator
g = torch.Generator().manual_seed(SEED)
train_set, val_set = random_split(train_full, [45000, 5000], generator=g)

# Per-channel mean/std from the 45k training images
X = torch.stack([train_set[i][0] for i in range(len(train_set))])  # (45000, 3, 32, 32)
mean = X.mean(dim=(0, 2, 3))
std  = X.std(dim=(0, 2, 3))
print("="*30, "mean/std", "="*30)
print("mean: ", mean.tolist())
print("std: ", std.tolist())

Files already downloaded and verified
Files already downloaded and verified
============================== mean/std ==============================
mean:  [0.4911963641643524, 0.48199522495269775, 0.4464317560195923]
std:  [0.24715185165405273, 0.24362300336360931, 0.26167726516723633]


## **Data Normalization**

In [12]:
def to_tensors(subset):
    xs = torch.stack([subset[i][0] for i in range(len(subset))])
    ys = torch.tensor([subset[i][1] for i in range(len(subset))])
    return (xs - mean[None, :, None, None]) / std[None, :, None, None], ys

Xtr, ytr = to_tensors(train_set)
Xva, yva = to_tensors(val_set)
Xte, yte = to_tensors(test_set)  
print("="*30, "shapes", "="*30)
print("Train: ", "Xtr: ", Xtr.shape, "ytr: ", ytr.shape)
print("Validation: ", "Xva: ", Xva.shape, "yva: ", yva.shape)
print("Test: ", "Xte: ", Xte.shape, "yte: ", yte.shape)

============================== shapes ==============================
Train:  Xtr:  torch.Size([45000, 3, 32, 32]) ytr:  torch.Size([45000])
Validation:  Xva:  torch.Size([5000, 3, 32, 32]) yva:  torch.Size([5000])
Test:  Xte:  torch.Size([10000, 3, 32, 32]) yte:  torch.Size([10000])


## **Task 01**
### 1(a): Layers with explicit forward / backward (no autograd)

In [21]:
class Linear:
    def __init__(self, n_in, n_out,init="kaiming", dtype=torch.float32, device="cpu"):
        # pick std for random weights based on init method
        if init == "kaiming":
            # for ReLU, std = sqrt(2/n_in)
            std = math.sqrt(2.0/n_in)
        elif init == "xavier":
            # for sigmoid, std = sqrt(2/(n_in+n_out))
            std = math.sqrt(2.0/(n_in+n_out))
        else:
            raise ValueError("Unknown init method: {}".format(init))
        
        self.W = torch.randn(n_out, n_in, dtype=dtype, device=device) * std
        self.b = torch.zeros(n_out, dtype=dtype, device=device)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)
        
    def forward(self, X):
        self.X = X
        return X @ self.W.T + self.b
    
    def backward(self, dY):
        self.dW = dY.T @ self.X
        self.db = dY.sum(dim=0)
        return dY @ self.W
    
    def params(self):
        return [self.W, self.b]
    def grads(self):
        return [self.dW, self.db]
        
class ReLU:
    def forward(self, X):
        self.mask = X>0
        return X * self.mask
    def backward(self, dY):
        return dY * self.mask
    def params(self):
        return []
    def grads(self):
        return []
    
class Sigmoid:
    def forward(self, X):
        self.out = 1/(1+torch.exp(-X))
        return self.out
    def backward(self, dY):
        return dY * self.out * (1-self.out)
    def params(self):
        return []
    def grads(self):
        return []
    
class SoftmaxCrossEntropy:
    def forward(self, logits, y):
        z = logits - logits.max(dim=1, keepdim=True).values
        log_p = z - torch.log(torch.exp(z).sum(dim=1, keepdim=True))
        self.p = torch.exp(log_p)
        self.y = y
        return -log_p[torch.arange(len(y)), y].mean()
    
    def backward(self):
        dlogits = self.p.clone()
        dlogits[torch.arange(len(self.y)), self.y] -= 1.0
        return dlogits / len(self.y)

### 1(a) Assembling the MLP

In [23]:
class MLP:
    def __init__(self, sizes, act = "relu", init = "kaiming", dtype=torch.float32, device="cpu"):
        self.layers = []
        for i in range(len(sizes)-1):
            self.layers.append(Linear(sizes[i], sizes[i+1], init, dtype, device))
            if i<len(sizes)-2:
                self.layers.append(ReLU() if act == "relu" else Sigmoid())
        
        self.loss_fn = SoftmaxCrossEntropy()
        
    def logits(self, X):
        for  layer in self.layers:
            X = layer.forward(X)
        return X
    
    def loss(self, X, y):
        return self.loss_fn.forward(self.logits(X), y)
    
    def backward(self):
        d = self.loss_fn.backward()
        for layer in reversed(self.layers):
            d = layer.backward(d)
        
    def params(self):
        return [p for l in self.layers for p in l.params()]
    def grads(self):
        return [g for l in self.layers for g in l.grads()]
    def linear_layers(self): 
        return [l for l in self.layers if isinstance(l, Linear)]

### 1(b) Gradient checking

In [25]:
def rel_err(a, b):
    return ((a - b).norm() / (a.norm() + b.norm() + 1e-30)).item()

def autograd_grads(net, X, y, act):
    """Same network, same weights, but built from torch ops and differentiated by autograd."""
    ps = [p.detach().clone().requires_grad_(True) for p in net.params()]
    n_lin = len(ps) // 2
    h = X
    for i in range(n_lin):
        h = h @ ps[2 * i].T + ps[2 * i + 1]
        if i < n_lin - 1:
            h = torch.relu(h) if act == "relu" else torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    loss.backward()
    return [p.grad for p in ps], loss.item()

def numeric_grads(net, X, y, eps=1e-6):
    grads = []
    for p in net.params():
        g = torch.zeros_like(p)
        flat, gflat = p.view(-1), g.view(-1)
        for i in range(flat.numel()):
            old = flat[i].item()
            flat[i] = old + eps; lp = net.loss(X, y).item()
            flat[i] = old - eps; lm = net.loss(X, y).item()
            flat[i] = old
            gflat[i] = (lp - lm) / (2 * eps)
        grads.append(g)
    return grads


# run a small test to check if the gradients are correct
names = [f"{n}{i+1}" for i in range(3) for n in ("W", "b")]      # W1,b1,W2,b2,W3,b3
rows = []
for act in ["relu", "sigmoid"]:
    set_seed(1)
    net = MLP([20, 16, 12, 10], act=act, dtype=torch.float64)
    for l in net.linear_layers():                                 # non-zero biases so db is tested properly
        l.b = torch.randn_like(l.b) * 0.1
    X = torch.randn(32, 20, dtype=torch.float64)
    y = torch.randint(0, 10, (32,))

    loss_mine = net.loss(X, y).item()
    net.backward()
    g_mine = [g.clone() for g in net.grads()]

    g_auto, loss_auto = autograd_grads(net, X, y, act)
    g_num = numeric_grads(net, X, y)

    print(f"[{act}] grad mine = {sum(g.norm() for g in g_mine):.12f}   autograd = {sum(g.norm() for g in g_auto):.12f}   numeric = {sum(g.norm() for g in g_num):.12f}")
    print(f"[{act}] loss mine = {loss_mine:.12f}   autograd = {loss_auto:.12f}")
    for n, a, b, c in zip(names, g_mine, g_auto, g_num):
        rows.append((act, n,
                     rel_err(a, b), torch.allclose(a, b, rtol=1e-6, atol=1e-8),
                     rel_err(a, c), torch.allclose(a, c, rtol=1e-4, atol=1e-7)))

print(f"\n{'act':8s}{'param':6s}{'relerr(auto)':>14s}{'allclose':>10s}{'relerr(FD)':>14s}{'allclose':>10s}")
for act, n, e1, ok1, e2, ok2 in rows:
    print(f"{act:8s}{n:6s}{e1:14.2e}{str(ok1):>10s}{e2:14.2e}{str(ok2):>10s}")

[relu] grad mine = 5.092469004132   autograd = 5.092469004132   numeric = 5.092469004432
[relu] loss mine = 3.237977674134   autograd = 3.237977674134
[sigmoid] grad mine = 1.136410894942   autograd = 1.136410894942   numeric = 1.136410895446
[sigmoid] loss mine = 2.505482963592   autograd = 2.505482963592

act     param   relerr(auto)  allclose    relerr(FD)  allclose
relu    W1          5.41e-17      True      1.44e-09      True
relu    b1          4.82e-17      True      4.95e-10      True
relu    W2          3.55e-17      True      8.41e-10      True
relu    b2          4.51e-17      True      6.83e-10      True
relu    W3          3.70e-17      True      5.87e-10      True
relu    b3          8.04e-18      True      1.10e-09      True
sigmoid W1          1.29e-16      True      2.00e-08      True
sigmoid b1          1.31e-16      True      1.77e-08      True
sigmoid W2          9.33e-17      True      4.89e-09      True
sigmoid b2          5.92e-17      True      2.54e-09      Tru